# MediGuide
## AI-Powered Analysis for Intelligent Healthcare Assistance

**Authors**
- Saurabh Kumbhar — 25204974
- Azim Hassan — 25203062

### Project Overview
MediGuide is a Retrieval-Augmented Generation (RAG) based healthcare assistance project.
It extracts medical information from PDF documents, splits the content into manageable
chunks, converts those chunks into embeddings, stores them in Pinecone, and uses an
OpenAI model through LangChain to answer questions from the retrieved context.

> **Important:** MediGuide is an academic/informational project. It must not be treated
> as a substitute for diagnosis, emergency care, or advice from a qualified healthcare professional.

## Pipeline

`Medical PDFs → Text Extraction → Cleaning → Chunking → Embeddings → Pinecone → Retrieval → OpenAI → Answer`

This notebook is organized so that every major stage of the RAG pipeline can be tested independently.

## 1. Environment and Project Setup

Before running this notebook:

1. Activate the `mediguide` Conda environment.
2. Install the packages from `requirements.txt`.
3. Create a `.env` file in the project root.
4. Add `OPENAI_API_KEY` and `PINECONE_API_KEY` to `.env`.
5. Place the medical PDF files inside the `data/` directory.

Example `.env` structure:

```text
OPENAI_API_KEY=your_openai_api_key
PINECONE_API_KEY=your_pinecone_api_key
```

Never commit the `.env` file to GitHub.

In [8]:
# Display the current working directory.
# This is useful for confirming where Jupyter started the notebook.
from pathlib import Path

print("Current working directory:", Path.cwd())

Current working directory: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim


In [9]:
# If the notebook is inside a subfolder such as `research/` or `notebooks/`,
# this block attempts to locate the project root by looking for requirements.txt.

from pathlib import Path

current_path = Path.cwd()

if (current_path / "requirements.txt").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "requirements.txt").exists():
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

Project root: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim
Data directory: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim\data


## 2. Import Required Libraries

In [10]:
# Standard library
import os
from typing import List

# Environment variables
from dotenv import load_dotenv

# LangChain document processing
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Pinecone
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# OpenAI and RAG chain
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

print("Imports loaded successfully.")

Imports loaded successfully.


## 3. Load Medical PDF Documents

`DirectoryLoader` scans the `data/` folder and `PyPDFLoader` extracts text from each PDF page.
Keeping document loading in its own function makes the pipeline easier to reuse later from the Flask application.

In [38]:
def load_pdf_files(data_directory: Path) -> List[Document]:
    """Load all PDF files from the supplied directory.

    Parameters
    ----------
    data_directory:
        Directory containing the medical PDF documents.

    Returns
    -------
    List[Document]
        LangChain Document objects extracted from the PDFs.
    """
    if not data_directory.exists():
        raise FileNotFoundError(
            f"Data directory not found: {data_directory}\n"
            "Create the directory and place your medical PDF files inside it."
        )

    loader = DirectoryLoader(
        str(data_directory),
        glob="*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True,
    )

    return loader.load()

In [12]:
# Load the medical knowledge-base documents.
extracted_data = load_pdf_files(DATA_DIR)

print(f"Pages/documents extracted: {len(extracted_data)}")

100%|██████████| 1/1 [00:40<00:00, 40.56s/it]

Pages/documents extracted: 637


In [13]:
# Preview one extracted document without printing the entire dataset.
if extracted_data:
    print("Source:", extracted_data[0].metadata.get("source"))
    print("\nText preview:\n")
    print(extracted_data[0].page_content[:1000])

Source: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim\data\Medical_book.pdf

Text preview:




## 4. Keep Only Essential Metadata

PDF loaders may attach several metadata fields. For this project, only the source file is
required for basic traceability, so the documents are normalized before chunking.

In [14]:
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """Return documents containing only their text and source metadata."""

    minimal_docs = []

    for doc in docs:
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": doc.metadata.get("source", "unknown")},
            )
        )

    return minimal_docs


minimal_docs = filter_to_minimal_docs(extracted_data)

print(f"Documents after metadata cleanup: {len(minimal_docs)}")

Documents after metadata cleanup: 637


## 5. Split Documents into Chunks

Large documents should not be sent to the model as one block. Chunking creates smaller,
overlapping sections that can be embedded and retrieved independently.

Current configuration:
- **Chunk size:** 500 characters
- **Chunk overlap:** 20 characters

In [15]:
def split_documents(
    documents: List[Document],
    chunk_size: int = 500,
    chunk_overlap: int = 20,
) -> List[Document]:
    """Split documents into smaller overlapping text chunks."""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    return text_splitter.split_documents(documents)


text_chunks = split_documents(minimal_docs)

print(f"Number of text chunks created: {len(text_chunks)}")

Number of text chunks created: 5859


In [16]:
# Preview the first chunk.
if text_chunks:
    print("Source:", text_chunks[0].metadata.get("source"))
    print("\nChunk preview:\n")
    print(text_chunks[0].page_content)

Source: c:\Users\Shree\Desktop\MediGuide\projects-saurabh-azim\data\Medical_book.pdf

Chunk preview:

The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
